<title = "Sistema de recomendacion"></title>
<h1>Sistema de recomendacion</h1>
Autor: Adrián Robles Arques

# Sprint 1

Conociendo el Dataset de MovieLens, para comprobar que se puede realizar una recomendación de películas a los usuarios.

In [5]:
# Importamos las librerias
import pandas as pd 
import numpy as np

In [6]:
# Cargamos el dataset
peliculas = pd.read_csv('datos\\movies.csv')

peliculas.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
# Cambiamos el nombre de las columnas
peliculas.columns = ['PeliculaID', 'titulo', 'generos']

peliculas.head()

,PeliculaID,titulo,generos
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
# Análisis descriptivo

peliculas.describe()

,PeliculaID
count,9742.000000
mean,42200.353623
std,52160.494854
min,1.000000
25%,3248.250000
50%,7300.000000
75%,76232.000000
max,193609.000000


In [9]:
# Importamos el dataset de ratings
ratings = pd.read_csv('datos\\ratings.csv')

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [10]:
# Cambiamos el nombre de las columnas
ratings.columns = ['UsuarioID', 'PeliculaID', 'Valoracion', 'Tiempo']

ratings.head()

,UsuarioID,PeliculaID,Valoracion,Tiempo
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [11]:
ratings.describe()

,UsuarioID,PeliculaID,Valoracion,Tiempo
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


# Primer intento de SR

In [12]:
# SR por popularidad

# Contamos el número de valoraciones por película
conteo_valoraciones = ratings.groupby('PeliculaID').size().reset_index(name='num_valoraciones')

# Unimos el conteo con el dataframe de películas y lo ordenamos
Peliculas_populares = pd.merge(peliculas, conteo_valoraciones, on='PeliculaID', how='left')
Peliculas_populares.set_index('PeliculaID', inplace=True)
Peliculas_populares.sort_values(by='num_valoraciones', ascending=False, inplace=True)
Peliculas_populares['num_valoraciones'] = Peliculas_populares['num_valoraciones'].astype('Int64')
Peliculas_populares.head(10)


,titulo,generos,num_valoraciones
PeliculaID,,,
356,Forrest Gump (1994),Comedy|Drama|Romance|War,329
318,"Shawshank Redemption, The (1994)",Crime|Drama,317
296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,307
593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,279
2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,278
260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,251
480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,238
110,Braveheart (1995),Action|Drama|War,237
589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,224


# Segundo intento de SR

Vamos a considerar las notas

In [13]:
# Vamos a contabilizar el promedio de valoraciones por película
promedio_valoraciones = ratings.groupby('PeliculaID')['Valoracion'].mean().reset_index(name='ValoracionPromedio')

# Vamos a unir los dataframes y ordenar la valoración promedio
Peliculas_valoradas = pd.merge(peliculas, promedio_valoraciones, on='PeliculaID', how='left')
Peliculas_valoradas.set_index('PeliculaID', inplace=True)
Peliculas_valoradas.sort_values(by='ValoracionPromedio', ascending=False, inplace=True)
Peliculas_valoradas['ValoracionPromedio'] = Peliculas_valoradas['ValoracionPromedio'].astype('float64').round(3)
Peliculas_valoradas.head(10)

# Vamos a unir los dos dataframes para ver las películas más populares y mejor valoradas
Peliculas_populares_valoradas = pd.merge(Peliculas_populares, Peliculas_valoradas[['ValoracionPromedio']], left_index=True, right_index=True, how='left')
Peliculas_populares_valoradas.sort_values(by='num_valoraciones', ascending=False, inplace=True)
Peliculas_populares_valoradas.head(10)

,titulo,generos,num_valoraciones,ValoracionPromedio
PeliculaID,,,,
356,Forrest Gump (1994),Comedy|Drama|Romance|War,329,4.164
318,"Shawshank Redemption, The (1994)",Crime|Drama,317,4.429
296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,307,4.197
593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,279,4.161
2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,278,4.192
260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,251,4.231
480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,238,3.750
110,Braveheart (1995),Action|Drama|War,237,4.032
589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,224,3.971


In [14]:
# Comparamos con el ordenamiento por valoración promedio
Peliculas_populares_valoradas.sort_values(by='ValoracionPromedio', ascending=False, inplace=True)
Peliculas_populares_valoradas.head(10)

,titulo,generos,num_valoraciones,ValoracionPromedio
PeliculaID,,,,
4116,Hollywood Shuffle (1987),Comedy,1,5.0
1140,Entertaining Angels: The Dorothy Day Story (1996),Drama,1,5.0
131610,Willy/Milly (1986),Comedy|Fantasy,1,5.0
131724,The Jinx: The Life and Deaths of Robert Durst ...,Documentary,1,5.0
141718,Deathgasm (2015),Comedy|Horror,1,5.0
138966,Nasu: Summer in Andalusia (2003),Animation,1,5.0
130970,George Carlin: Life Is Worth Losing (2005),Comedy,1,5.0
141816,12 Chairs (1976),Adventure|Comedy,1,5.0
142020,Oscar (1967),Comedy,1,5.0


In [15]:
# Filtramos por películas con más de 10 valoraciones
Peliculas_populares_valoradas.query('num_valoraciones >= 50', inplace=True)
Peliculas_populares_valoradas.sort_values(by='ValoracionPromedio', ascending=False, inplace=True)
Peliculas_populares_valoradas.head(10)

C:\Users\demad\AppData\Local\Temp\ipykernel_16008\2147617893.py:2: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  Peliculas_populares_valoradas.query('num_valoraciones >= 50', inplace=True)


,titulo,generos,num_valoraciones,ValoracionPromedio
PeliculaID,,,,
318,"Shawshank Redemption, The (1994)",Crime|Drama,317,4.429
858,"Godfather, The (1972)",Crime|Drama,192,4.289
2959,Fight Club (1999),Action|Crime|Drama|Thriller,218,4.273
1276,Cool Hand Luke (1967),Drama,57,4.272
750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,97,4.268
904,Rear Window (1954),Mystery|Thriller,84,4.262
1221,"Godfather: Part II, The (1974)",Crime|Drama,129,4.260
48516,"Departed, The (2006)",Crime|Drama|Thriller,107,4.252
1213,Goodfellas (1990),Crime|Drama,126,4.250


# Tercer intento de SR

Recomendación en función de los gustos del usuario.

# SPRINT 2

Cálculo de distancias entre usuarios

In [16]:
# Definido un espacio vectorial para la calificación de películas
# Vamos a dotarlo de una métrica g = [Sum_i(dx_i^2)]^(1/2)
# Donde dx_i es la diferencia entre la valoración del usuario objetivo y el usuario i

juan = np.array([5,5])
sergio = np.array([4,4.5])

resta = juan - sergio
distancia = np.sqrt(np.sum(resta**2))
print(f"La distancia entre Juan y Sergio es: {distancia:.2f}")

# Vamos a construir la función de distancia

def dist_euclidia(x, y):
    """
    Calcula la distancia euclidiana entre dos vectores x e y.
    """
    x = np.array(x)
    y = np.array(y)
    resta = x - y
    return np.sqrt(np.sum(resta**2))

La distancia entre Juan y Sergio es: 1.12


In [17]:
# Vamos a mejorar la función para que reciba una lista de vectores de cualquier dimensión
def N_dist_euclidia(v_objetivo, vectores):
    """
    Calcula la distancia euclidiana entre un vector objetivo y una lista de vectores.
    
    :param v_objetivo: Vector objetivo (array-like).
    :param vectores: Lista de vectores (array-like).
    :return: Lista de distancias euclidianas.
    """
    v_objetivo = np.array(v_objetivo)
    distancias = []
    for v in vectores:
        v = np.array(v)
        distancias.append(np.linalg.norm(v_objetivo - v))
        
    return distancias

In [18]:
ratings.head()

,UsuarioID,PeliculaID,Valoracion,Tiempo
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [19]:
usuario1 = ratings.query('UsuarioID==1')[['PeliculaID', 'Valoracion']]
usuario1.head(10)

,PeliculaID,Valoracion
0,1,4.0
1,3,4.0
2,6,4.0
3,47,5.0
4,50,5.0
5,70,3.0
6,101,5.0
7,110,4.0
8,151,5.0
9,157,5.0


In [20]:
usuario4 = ratings.query('UsuarioID==4')[['PeliculaID', 'Valoracion']]
usuario4.head(10)

,PeliculaID,Valoracion
300,21,3.0
301,32,2.0
302,45,3.0
303,47,2.0
304,52,3.0
305,58,3.0
306,106,4.0
307,125,5.0
308,126,1.0
309,162,5.0


In [21]:
usuarios1_4 = pd.merge(usuario1, usuario4, left_on='PeliculaID', right_on='PeliculaID', how='inner', suffixes=('_u1','_u4'))
usuarios1_4.head()

,PeliculaID,Valoracion_u1,Valoracion_u4
0,47,5.0,2.0
1,235,4.0,2.0
2,260,5.0,5.0
3,296,3.0,1.0
4,441,4.0,1.0


In [22]:
dist_euclidia(usuarios1_4['Valoracion_u1'], usuarios1_4['Valoracion_u4'])

11.135528725660043

In [101]:
# Vamos a definir la función para calcular la distancia entre dos usuarios cualesquiera tal como lo hemos definido

def dist_usuarios(user1, user2, minimo_peliculas=5, ratings=ratings):
    """Calcula la distancia euclidiana entre dos usuarios basándose en las valoraciones de películas que ambos han visto.
    
    Args:
        user1: ID del primer usuario. Int
        user2: ID del segundo usuario. Int
        minimo_peliculas: Número mínimo de películas que ambos usuarios deben haber visto para calcular la distancia. Int
        ratings: DataFrame de valoraciones de películas. DataFrame[UsuarioID, PeliculaID, Valoracion, Tiempo]
    
    Returns:
        La Distancia euclidiana entre los dos usuarios o None si no hay suficientes películas en común.
    """
    valoracion_u1 = ratings.query('UsuarioID==%d' % user1)[['PeliculaID', 'Valoracion']]
    valoracion_u2 = ratings.query('UsuarioID==%d' % user2)[['PeliculaID', 'Valoracion']]
    
    # Creamos un dataframe con las dos series de notas para las películas que hayan visto ambos
    valoracion_u1_u2 = pd.merge(valoracion_u1, valoracion_u2, on='PeliculaID', how='inner', suffixes=('_u1', '_u2'))
    
    if len(valoracion_u1_u2) < minimo_peliculas:
        return None
    
    # Extraemos los vectores
    x = np.array(valoracion_u1_u2['Valoracion_u1'])
    y = np.array(valoracion_u1_u2['Valoracion_u2'])
    
    # Calculamos la distancia entre los dos usuarios
    return np.linalg.norm(x - y)
    
# Vamos a definir la función para calcular la distancia entre dos usuarios cualesquiera tal como lo hemos definido()


In [102]:
dist_usuarios(5,4)

6.324555320336759

In [103]:
# Ahora vamos a encontrar los usuarios más próximos a un usuario dado, para ello necesitamos generalizar para N

# Primero vamos a extraer todos los ID de los usuarios únicos
usuarios_tot = ratings['UsuarioID'].unique()
usuarios_tot

array([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [110]:
def k_mas_similares(user_obj, usuarios, us_similares=1, minimo_peliculas=5, ratings=ratings):
    """
    Retorna el usuario mas similar al usuario ingresado
    
    Args:
        user_obj: Objeto de usuario
        usuarios: Lista de usuarios
        us_similares: Número de usuarios similares a retornar, por defecto 1
        Para dist_usuarios:
            minimo_peliculas: Número mínimo de películas compartidas. Int
            ratings: DataFrame de valoraciones
    Returns:
        Usuarios mas similarres al usuario ordenador y la distancia
    """
    # Excluimos el usuario objetivo de la lista de usuarios
    usuarios_ex = list(usuarios.copy())
    usuarios_ex.remove(user_obj)
    
    distancias = {}
    for usuario in usuarios_ex:
        dist = dist_usuarios(user_obj, usuario, minimo_peliculas=minimo_peliculas, ratings=ratings)
        distancias[usuario] = dist
    
    # Eliminamos aquellos que tengan valor None
    distancias = {k: v for k, v in distancias.items() if v is not None}
    
    # Ordenamos la lista de usuarios por su distancia
    usuarios_ordenados = np.array(sorted(distancias.items(), key=lambda x: x[1])).reshape(-1, 2)
    result = pd.DataFrame(usuarios_ordenados[:us_similares], columns=['UsuarioID', 'Distancia_user_ref'])
    result['UsuarioID'] = result['UsuarioID'].astype('Int64')
    result.set_index('UsuarioID', inplace=True)
    
    return result


In [111]:
# Convertimos usuarios_tot a lista para evitar problemas con remove
usuario_sim = k_mas_similares(1, usuarios_tot, 3)
usuario_sim


,Distancia_user_ref
UsuarioID,
77,0.000000
511,0.500000
366,0.707107


In [112]:
usuarios_sim_1 = k_mas_similares(1, usuarios_tot, len(usuarios_tot))
usuarios_sim_1

,Distancia_user_ref
UsuarioID,
77,0.000000
511,0.500000
366,0.707107
9,1.000000
49,1.000000
...,...
474,18.594354
160,18.794946
217,19.646883


# Sprint 3

Identificando al usuario más próximo con otras métricas

In [113]:
# Lo que hace al principio del sprint ya lo hice en la función "usuario_mas_similar"
# Vamos a exraer las películas vistas por los usuarios similares que el usuario objetivo no ha visto

def peliculas_no_vistas_por_usuario(user_obj, usuarios_similares):
    """
    Retorna las películas que han visto los usuarios similares al usuario objetivo y que este no ha visto.
    
    Args:
        user_obj: ID del usuario objetivo.
        usuarios_similares: DataFrame con los usuarios similares y sus distancias.
        
    Returns:
        DataFrame con las películas no vistas por el usuario objetivo.
    """
    # Extraemos las películas vistas por el usuario objetivo
    peliculas_vistas = ratings.query('UsuarioID==%d' % user_obj)['PeliculaID'].unique()
    
    # Creamos un DataFrame para almacenar las películas no vistas
    peliculas_no_vistas = pd.DataFrame()
    
    for usuario in usuarios_similares.index:
        # Extraemos las películas vistas por el usuario similar
        peliculas_usuario = ratings.query('UsuarioID==%d' % usuario)[['PeliculaID', 'Valoracion']]
        
        # Filtramos las películas que el usuario objetivo no ha visto
        peliculas_usuario_no_vistas = peliculas_usuario[~peliculas_usuario['PeliculaID'].isin(peliculas_vistas)]
        
        # Añadimos al DataFrame de películas no vistas
        peliculas_no_vistas = pd.concat([peliculas_no_vistas, peliculas_usuario_no_vistas])
    
    return peliculas_no_vistas.reset_index(drop=True)

In [118]:
# Vamos a recomendar las películas no vistar por el usuario objetivo
# Vamos a probar a añadir filtrado por género
def recomendar_peliculas(user_obj, usuarios, peliculas, usuarios_similares = 5, num_recomendaciones=5, Generos=None, 
                        minimo_peliculas=5, ratings=ratings):
    """
    Recomienda películas al usuario objetivo basándose en las valoraciones de usuarios similares.
    
    Args:
        user_obj: ID del usuario objetivo. Int
        usuarios: Lista de IDs de usuarios. Array-like(Int)
        peliculas: DataFrame con las películas y sus géneros. DataFrame
        usuarios_similares: Cantidad de usuarios similares a considerar. Int
        Generos: Lista de géneros a filtrar las recomendaciones. Array-like(String)
        num_recomendaciones: Número de recomendaciones a retornar.
        minimo_peliculas: Número mínimo de películas que deben haber visto los usuarios similares. Int
        ratings: DataFrame de valoraciones de películas. DataFrame[UsuarioID, PeliculaID, Valoracion, Tiempo]
    
    Returns:
        DataFrame con las películas recomendadas y sus valoraciones promedio.
    """
    usr_sim = k_mas_similares(user_obj, usuarios, usuarios_similares,
                                minimo_peliculas=minimo_peliculas, ratings=ratings)
    
    peliculas_no_vistas = peliculas_no_vistas_por_usuario(user_obj, usr_sim)
    
    # Agrupamos por PeliculaID y calculamos la valoración promedio
    recomendaciones = peliculas_no_vistas.groupby('PeliculaID')['Valoracion'].mean().reset_index()
    
    # Ordenamos por valoración promedio y seleccionamos las mejores
    recomendaciones.sort_values(by='Valoracion', ascending=False, inplace=True)
    recomendaciones.set_index('PeliculaID', inplace=True)
    
    
    peliculas_recomendadas = pd.merge(recomendaciones,
                                    peliculas.reset_index()[['PeliculaID','titulo', 'generos']],
                                    left_index=True, right_on='PeliculaID', how='left')
    
    # Filtramos por géneros si se especifican
    if Generos:
        # El método 'apply' se usa para aplicar una función a cada fila del DataFrame que genera una máscara booleana
        # que indica si al menos uno de los géneros de la película está en la lista
        mask = peliculas_recomendadas.apply(lambda x: any(g in Generos for g in x['generos'].split('|')), axis=1)
        
        # La máscara es un array booleano que indica si cada fila cumple la condición
        # Aplicamos la máscara para filtrar las películas recomendadas
        peliculas_recomendadas = peliculas_recomendadas[mask]
    
    # Devolvemos la cantidad fijada de recomendaciones
    peliculas_recomendadas = peliculas_recomendadas.head(num_recomendaciones)
    return peliculas_recomendadas[['titulo', 'generos', 'Valoracion']].reset_index(drop=True)

Tras propagar hasta la última función todos los argumentos opcionales de las funciones previas requeridas, ahora se puede controlar desde la función de recomendación todas las variables y datos empleadas en la misma.

In [119]:
recomendar_peliculas(421, usuarios_tot, peliculas ,num_recomendaciones=10, Generos=['Fantasy', 'Adventure'])


,titulo,generos,Valoracion
0,300 (2007),Action|Fantasy|War|IMAX,5.0
1,Stand by Me (1986),Adventure|Drama,5.0
2,"Nightmare Before Christmas, The (1993)",Animation|Children|Fantasy|Musical,5.0
3,Edward Scissorhands (1990),Drama|Fantasy|Romance,5.0
4,Being John Malkovich (1999),Comedy|Drama|Fantasy,5.0
5,Star Trek: First Contact (1996),Action|Adventure|Sci-Fi|Thriller,5.0
6,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,5.0
7,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,5.0
8,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,5.0
9,Fantastic Mr. Fox (2009),Adventure|Animation|Children|Comedy|Crime,5.0


# Intento propio

Recomendador comprobando gustos por género de cada usuario, con métrica definida

In [90]:
#Separamos los géneros en listas
# Primero vamos a crear una copia del dataframe original
peliculas_rec = peliculas.copy()
peliculas_rec['generos'] = peliculas['generos'].str.split('|')

#Hacemos la codificación one_hot manualmente dado que trabajamos con una lista de géneros

#Primero vamos a extraer la lista del total de géneros diferentes
genres_list = []
for index, row in peliculas_rec.iterrows():
    for genre in row['generos']:
        if genre not in genres_list:
            genres_list.append(genre)

#Ahora vamos a iterar por género, comprobando si aparece en la lista de géneros de la película
for genre in genres_list:
    aux_list = []
    for index, row in peliculas_rec.iterrows():
        aux_list.append(int(genre in row['generos']))
    peliculas_rec[genre] = aux_list

peliculas_rec.head()

,PeliculaID,titulo,generos,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),[Comedy],0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [91]:
# Vamos a encontrar las películas que son del género (No Genero)
peliculas_rec.query('`(no genres listed)` == 1')

# Aquí tenemos películas que no tienen género asignado, lo cual es un caso especial.

,PeliculaID,titulo,generos,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
8517,114335,La cravate (1957),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8684,122888,Ben-hur (2016),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8782,129250,Superfast! (2015),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8836,132084,Let It Be Me (1995),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8902,134861,Trevor Noah: African American (2013),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9033,141131,Guardians (2016),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9053,141866,Green Room (2015),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9070,142456,The Brand New Testament (2015),[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9091,143410,Hyena Road,[(no genres listed)],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [93]:
# Vamos a eliminar del Dataframe las películas que no tienen género
peliculas_rec = peliculas_rec[peliculas_rec['(no genres listed)'] != 1]

# Eliminamos la columna de (no genres listed) ya que no es necesaria
peliculas_rec = peliculas_rec.drop(columns=['(no genres listed)'])


peliculas_rec.head()

,PeliculaID,titulo,generos,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),[Comedy],0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [95]:
# Ahora vamos a eliminar 'no genres listed' de la lista de géneros
genres_list.remove('(no genres listed)')

In [96]:
# Ahora vamos a generar un vetor normal de los géneros

peliculas_rec['vector_gen'] = peliculas_rec[genres_list].values.tolist()
peliculas_rec['vector_gen'] = peliculas_rec['vector_gen'].apply(lambda x: np.array(x) / np.linalg.norm(x))
peliculas_rec.set_index('PeliculaID', inplace=True)

peliculas_rec.head()

,titulo,generos,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,vector_gen
PeliculaID,,,,,,,,,,,,,,,,,,,,,
1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,"[0.4472135954999579, 0.4472135954999579, 0.447..."
2,Jumanji (1995),"[Adventure, Children, Fantasy]",1,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,"[0.5773502691896258, 0.0, 0.5773502691896258, ..."
3,Grumpier Old Men (1995),"[Comedy, Romance]",0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,"[0.0, 0.0, 0.0, 0.7071067811865475, 0.0, 0.707..."
4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,0,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,"[0.0, 0.0, 0.0, 0.5773502691896258, 0.0, 0.577..."
5,Father of the Bride Part II (1995),[Comedy],0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


A partir del vector generador, que es unitario, podemos emplear diferentes métodos de clasificación para realizar un algoritmo de recomendación.

* Similitud del coseni
* Distancia euclidea
* Algoritmo de K-means